# Delete NWM v3.0 streamflow written after the v3.1 cutover (#439)

Until #396 the ingests ran as `nwm30`, so the rows they wrote after the v3.1 cutover
(2026-08-18 t00z) are labelled `nwm30_*`. Run this only after
`02_loading/23_load_backfill_nwm31_streamflow.ipynb` has loaded and checked the
replacement `nwm31_*` data.

Forecast configurations are filtered on `reference_time`, so v3.0 forecasts issued before
the cutover are kept, including medium range cycles whose value times run past it. Analysis
configurations have a NULL `reference_time`, so they are filtered on `value_time`.

In [ ]:
import teehr
from teehr.models.filters import TableFilter
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr.fetching.utils import format_nwm_configuration_metadata

teehr.__version__

In [ ]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=8,
    executor_memory="32g",
    executor_cores=4,
    aws_profile="admin-user"
)

In [ ]:
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

In [ ]:
CUTOVER = "2026-08-18 00:00:00"

CONFIGS = [
    ("short_range", "channel_rt"),
    ("short_range_alaska", "channel_rt"),
    ("short_range_hawaii", "channel_rt"),
    ("short_range_puertorico", "channel_rt"),
    ("medium_range_mem1", "channel_rt_1"),
    ("medium_range_blend", "channel_rt"),
    ("medium_range_alaska_mem1", "channel_rt_1"),
    ("medium_range_blend_alaska", "channel_rt"),
    ("analysis_assim_no_da", "channel_rt"),
    ("analysis_assim_alaska_no_da", "channel_rt"),
    ("analysis_assim_extend_no_da", "channel_rt"),
    ("analysis_assim_extend_alaska_no_da", "channel_rt"),
    ("analysis_assim_hawaii_no_da", "channel_rt"),
    ("analysis_assim_puertorico_no_da", "channel_rt"),
]

config_names = [
    format_nwm_configuration_metadata(c, "nwm30")["name"] for c, _ in CONFIGS
]


def cutover_column(config_name):
    return "value_time" if "analysis" in config_name else "reference_time"


def cutover_filters(config_name):
    return [
        TableFilter(column="configuration_name", operator="=", value=config_name),
        TableFilter(column=cutover_column(config_name), operator=">=", value=CUTOVER),
    ]

#### Preview
Row counts per configuration, using each one's cutover column.

In [ ]:
%%time
names_sql = ", ".join(f"'{n}'" for n in config_names)
cutover_sql = " ".join(
    f"WHEN '{n}' THEN {cutover_column(n)}" for n in config_names
)
ev.spark.sql(f"""
    SELECT
        configuration_name,
        COUNT_IF(CASE configuration_name {cutover_sql} END >= '{CUTOVER}') AS n_to_delete,
        MIN(reference_time) AS min_reference_time,
        MAX(reference_time) AS max_reference_time,
        MIN(value_time) AS min_value_time,
        MAX(value_time) AS max_value_time
    FROM iceberg.teehr.secondary_timeseries
    WHERE configuration_name IN ({names_sql})
    GROUP BY configuration_name
    ORDER BY configuration_name
""").toPandas()

#### Delete one configuration at a time

In [ ]:
config_name = "nwm30_short_range"
filters = cutover_filters(config_name)

In [ ]:
# Optional preview of rows that will be deleted
rows_to_delete = ev.secondary_timeseries.delete(
    filters=filters,
    dry_run=True,
)
print(f"rows to delete: {rows_to_delete.count()}")

In [ ]:
rows_to_delete.show()

In [ ]:
# Delete the post-cutover slice
deleted_count = ev.secondary_timeseries.delete(filters=filters)
print(f"deleted rows: {deleted_count}")

In [ ]:
ev.secondary_timeseries.filter(filters).to_sdf().count()

In [ ]:
spark.stop()